# 03 - GraphSAGE - inductive, sampled, scalable

GCN needs the **whole** graph (that full $\hat A$ matrix) and is **transductive**: it
learns fixed embeddings for the exact nodes it trained on. Real graphs are huge and
*grow* (new users, new papers). **GraphSAGE** (Hamilton et al., 2017) fixes both:

- **Separate self & neighbour transforms**, then aggregate: $h_i' = \sigma\big(W\,[\,h_i \,\Vert\, \text{AGG}_{j\in N(i)} h_j\,]\big)$ - it *learns a function*, so it works on **unseen** nodes (inductive).
- **Neighbour sampling**: aggregate over a fixed-size random sample of neighbours, so cost
  is bounded regardless of hub degree -> it scales to graphs with millions of nodes.

In [1]:
import os, sys, warnings, time
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))   # make utils/ importable from notebooks/

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
torch.manual_seed(0); np.random.seed(0)

from utils import graphs as G     # synthetic graph generators (known ground truth)
from utils import models as M     # the model zoo (MLP, GCN, SAGE, GAT, GIN, GPS, ...)
from utils import training as T    # train/eval loops + metrics
from utils import plotting as P    # graph drawing, curves, comparisons


## Aggregator choice matters

SAGE can aggregate with `mean`, `max`, etc. Mean is a smooth summary; max picks out the
single most salient neighbour signal. PyG exposes this via `SAGEConv(aggr=...)`.

In [2]:
data, gt = G.make_sbm_homophily(homophily=0.8, seed=1)
from torch_geometric.nn import SAGEConv
import torch.nn as nn, torch.nn.functional as Fnn

class SAGE(nn.Module):
    def __init__(self, ind, hid, outd, aggr="mean"):
        super().__init__()
        self.c1 = SAGEConv(ind, hid, aggr=aggr); self.c2 = SAGEConv(hid, outd, aggr=aggr)
    def forward(self, x, ei):
        return self.c2(Fnn.relu(self.c1(x, ei)), ei)

C = int(data.y.max())+1
for aggr in ["mean", "max"]:
    T.set_seed(0)
    res = T.train_node(SAGE(data.num_features,32,C,aggr), data, epochs=200)
    print(f"SAGE aggr={aggr:4s}  test acc = {res['test_metric']:.3f}")

SAGE aggr=mean  test acc = 0.978


SAGE aggr=max   test acc = 0.944


## Inductive learning: predict nodes you never trained on

Because SAGE learns a *function* of (self, neighbourhood) rather than a fixed
per-node embedding, it generalises to **unseen** nodes. We make this airtight: train
**only on the subgraph induced by the training nodes** - the model literally never sees
the test nodes or their edges during training - then run inference on the full graph.

In [3]:
from torch_geometric.utils import subgraph

big, _ = G.make_sbm_homophily(n_per_block=300, n_blocks=3, homophily=0.8, seed=2)
C = int(big.y.max())+1
train_nodes = big.train_mask.nonzero(as_tuple=True)[0]

# Induced training subgraph: edges only among training nodes, relabelled 0..n_tr-1.
tr_ei, _ = subgraph(train_nodes, big.edge_index, relabel_nodes=True, num_nodes=big.num_nodes)
tr_x, tr_y = big.x[train_nodes], big.y[train_nodes]

T.set_seed(0)
model = SAGE(big.num_features, 32, C)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(150):
    model.train(); opt.zero_grad()
    loss = Fnn.cross_entropy(model(tr_x, tr_ei), tr_y)
    loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    pred = model(big.x, big.edge_index).argmax(1)        # full-graph inductive inference
acc = (pred[big.test_mask] == big.y[big.test_mask]).float().mean().item()
print(f"trained on {len(train_nodes)} nodes only; "
      f"inductive acc on {int(big.test_mask.sum())} never-seen nodes = {acc:.3f}")

trained on 540 nodes only; inductive acc on 180 never-seen nodes = 0.978


## Neighbour sampling: the scaling trick, in a few lines

On a graph with million-degree hubs, aggregating over *all* neighbours every step is
hopeless. SAGE instead aggregates over a **fixed-size random sample** - bounding the
work per node while keeping a faithful summary. (PyG's `NeighborLoader` does this
efficiently with a compiled sampler; here is the idea by hand.)

In [4]:
from collections import defaultdict
adj = defaultdict(list)
for u, v in big.edge_index.t().tolist():
    adj[v].append(u)

def sampled_neighbor_mean(node, x, k=8):
    nb = adj[node]
    if len(nb) > k:
        nb = list(np.random.choice(nb, k, replace=False))
    return x[nb].mean(0) if nb else x[node]

hub = max(adj, key=lambda nd: len(adj[nd]))              # the highest-degree node
full = big.x[adj[hub]].mean(0)
samp = sampled_neighbor_mean(hub, big.x, k=8)
sim = Fnn.cosine_similarity(full, samp, dim=0).item()
print(f"hub node {hub} has {len(adj[hub])} neighbours.")
print(f"Sampling just 8 of them keeps cosine similarity {sim:.2f} to the full average,")
print("at a cost that no longer depends on the hub's degree.")

hub node 173 has 20 neighbours.
Sampling just 8 of them keeps cosine similarity 0.96 to the full average,
at a cost that no longer depends on the hub's degree.


## When to reach for GraphSAGE

- (+) **Large graphs** where the full adjacency won't fit / is too slow - sampling bounds cost.
- (+) **Inductive** settings: nodes arrive after training (new users/items/papers).
- (+) When the **node's own features** must stay distinct from the neighbourhood summary
  (recall GCN's blending problem in notebook 02).
- The aggregator (`mean`/`max`/`lstm`) is a real knob - tune it.

Next: **GAT**, which learns *how much to listen to each neighbour* instead of averaging
them equally.